# 13. The Final Scrub: Complete Data Cleaning Workflow

## Congratulations!

You've made it to the end of the data cleaning module! You now have the fundamental skills to take messy, real-world raw data and transform it into clean, analysis-ready DataFrames.

---

## Quick Recap: The 4 Main Data Cleaning Pillars

Before taking on the final project, here is a quick cheat sheet summarizing the four primary data quality issues and how to solve them in Pandas:

### 1. Missing Data

Incomplete rows can throw off calculations or break machine learning models.

```python
# Remove any row with missing data
cleaned_df = df.dropna()

# Remove rows with missing data in specific columns
cleaned_df = df.dropna(subset=['username', 'location'])

# Replace missing string values
df['location'] = df['location'].fillna('Unknown')

# Impute missing numerical values with the mean
mean_age = df['age'].mean()
df['age'] = df['age'].fillna(mean_age)

```

---

### 2. Duplicate Data

Repeated entries skew totals, averages, and statistics.

```python
# Identify duplicate rows (returns True/False series)
df.duplicated()

# Filter and view duplicate rows
df[df.duplicated()]

# Remove exact duplicate rows across all columns
no_dupes = df.drop_duplicates()

# Remove duplicate entries based on a unique identifier
unique_records = df.drop_duplicates(subset=['user_id'])

```

---

### 3. Wrong Data Types

Operations fail when numbers or booleans are incorrectly stored as text.

```python
# Inspect current data types across all columns
df.dtypes

# Convert text column to numeric (coercing errors to NaN)
df['age'] = pd.to_numeric(df['age'], errors='coerce')

```

---

### 4. Inconsistent Strings

User-entered text varies in casing, spacing, and phrasing.

```python
# Standardize text: convert to lower case and strip whitespace
responses['rsvp'] = responses['rsvp'].str.lower().str.strip()

# Replace specific substrings or values
responses['rsvp'] = responses['rsvp'].str.replace('yes', 'y')

```

---

## Final Project: Emergency Department Scrub

We've practiced each of these data cleaning tasks in isolation. In the real world, you'll apply all of these strategies together on a single dataset.

> **The Scenario:** Hospital emergency rooms (*inspired by shows like Grey's Anatomy, House, and The Pitt*) operate under fast-paced, high-pressure conditions. Because data is entered quickly and under stress, intake records are notoriously messy—full of missing values, duplicate entries, mismatched text casing, and incorrect data types.

You have been provided with an **Emergency Department Patient Intake DataFrame** that requires a full scrub before the medical director can analyze hospital efficiency.

---

### Suggested Cleaning Steps

There is no single "correct" sequence, but a standard cleaning pipeline often follows this flow:

```
[1. Inspect]       Check df.info(), df.dtypes, and df.head()
      ↓
[2. Deduplicate]   Remove duplicate logging errors with .drop_duplicates()
      ↓
[3. Standardize]   Clean string casing & trailing spaces with .str methods
      ↓
[4. Type Cast]     Convert numeric fields using pd.to_numeric(errors='coerce')
      ↓
[5. Handle NaNs]   Fill or drop missing values based on clinical context

```

### Your Tasks

1. **Inspect:** Examine the intake dataset to identify missing data, duplicates, and bad types.
2. **Deduplicate:** Remove redundant records representing system logging errors.
3. **Normalize Text:** Standardize medical priority categories (e.g., convert `'critical'`, `'CRITICAL'`, and `'Critical'` into a uniform `'critical'`).
4. **Fix Types:** Convert numerical intake measurements (e.g., heart rate, wait time) into numeric data types.
5. **Handle Missing Values:** Impute or drop empty fields based on clinical relevance.

In [2]:
import pandas as pd

data = {
        "patient_id": [
            "PT-2001",
            "PT-2001",
            "PT-2002",
            "PT-2003",
            "PT-2004",
            "PT-2005",
            "PT-2006",
            "PT-2006",
            "PT-2007",
            "PT-2008",
            "PT-2009",
            "PT-2010",
            "PT-2011",
        ],
        "name": [
            "Anthony Ramirez",
            "anthony ramirez",
            "Grace Mitchell",
            "Daniel Brooks",
            "Nina Patel",
            "Marcus Johnson",
            "Elena Cruz",
            "elena cruz",
            "William Carter",
            "Sophia Nguyen",
            None,
            "Sophia Nguyen",
            "Ethan Walker",
        ],
        "age": ["52", "52", 34, "28", "61", 47, "26", 26, "73", "39", 44, "39", "58"],
        "department": [
            "ER",
            "er",
            "Trauma",
            "ER",
            "Cardiology",
            "Trauma",
            "ER",
            "er",
            "ICU",
            "ER",
            "ER",
            "ER",
            "ICU",
        ],
        "triage_level": [
            "Critical",
            "critical",
            "Moderate",
            "HIGH",
            "Low",
            "Moderate",
            "critical",
            "CRITICAL",
            "High",
            "Low",
            "Moderate",
            "low",
            None,
        ],
        "wait_time_minutes": [
            "12",
            "12",
            55,
            "40 mins",
            140,
            60,
            "8",
            8,
            "95",
            "25",
            None,
            "25",
            "70",
        ],
        "admitted": [
            True,
            True,
            False,
            True,
            True,
            False,
            True,
            True,
            False,
            True,
            False,
            True,
            None,
        ],
}

patients = pd.DataFrame(data)
patients


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,ER,Critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,Trauma,Moderate,55,False
3,PT-2003,Daniel Brooks,28,ER,HIGH,40 mins,True
4,PT-2004,Nina Patel,61,Cardiology,Low,140,True
5,PT-2005,Marcus Johnson,47,Trauma,Moderate,60,False
6,PT-2006,Elena Cruz,26,ER,critical,8,True
7,PT-2006,elena cruz,26,er,CRITICAL,8,True
8,PT-2007,William Carter,73,ICU,High,95,False
9,PT-2008,Sophia Nguyen,39,ER,Low,25,True


In [5]:
patients.info()

patients.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   patient_id         13 non-null     object
 1   name               12 non-null     object
 2   age                13 non-null     object
 3   department         13 non-null     object
 4   triage_level       12 non-null     object
 5   wait_time_minutes  12 non-null     object
 6   admitted           12 non-null     object
dtypes: object(7)
memory usage: 860.0+ bytes


patient_id           object
name                 object
age                  object
department           object
triage_level         object
wait_time_minutes    object
admitted             object
dtype: object

In [8]:
patients['triage_level'] = patients['triage_level'].str.strip().str.lower()
patients['department'] = patients['department'].str.strip().str.lower()
patients

,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,er,critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,trauma,moderate,55,False
3,PT-2003,Daniel Brooks,28,er,high,40 mins,True
4,PT-2004,Nina Patel,61,cardiology,low,140,True
5,PT-2005,Marcus Johnson,47,trauma,moderate,60,False
6,PT-2006,Elena Cruz,26,er,critical,8,True
7,PT-2006,elena cruz,26,er,critical,8,True
8,PT-2007,William Carter,73,icu,high,95,False
9,PT-2008,Sophia Nguyen,39,er,low,25,True


In [9]:
patients = patients.dropna(subset = ['name'])
patients

,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,er,critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,trauma,moderate,55,False
3,PT-2003,Daniel Brooks,28,er,high,40 mins,True
4,PT-2004,Nina Patel,61,cardiology,low,140,True
5,PT-2005,Marcus Johnson,47,trauma,moderate,60,False
6,PT-2006,Elena Cruz,26,er,critical,8,True
7,PT-2006,elena cruz,26,er,critical,8,True
8,PT-2007,William Carter,73,icu,high,95,False
9,PT-2008,Sophia Nguyen,39,er,low,25,True


In [11]:
avg_admitted = patients['admitted'].mode().iloc[0]
avg_triage_level = patients['triage_level'].mode().iloc[0]

patients['admitted'] = patients['admitted'].fillna(avg_admitted)
patients['triage_level'] = patients['triage_level'].fillna(avg_triage_level)

patients


C:\Users\vesko\AppData\Local\Temp\ipykernel_21816\230550051.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patients['admitted'] = patients['admitted'].fillna(avg_admitted)
C:\Users\vesko\AppData\Local\Temp\ipykernel_21816\230550051.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patients['triage_level'] = patients['triage_level'].fillna(avg_triage_level)


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,er,critical,12,True
1,PT-2001,anthony ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,trauma,moderate,55,False
3,PT-2003,Daniel Brooks,28,er,high,40 mins,True
4,PT-2004,Nina Patel,61,cardiology,low,140,True
5,PT-2005,Marcus Johnson,47,trauma,moderate,60,False
6,PT-2006,Elena Cruz,26,er,critical,8,True
7,PT-2006,elena cruz,26,er,critical,8,True
8,PT-2007,William Carter,73,icu,high,95,False
9,PT-2008,Sophia Nguyen,39,er,low,25,True


In [13]:
patients_unique = patients.drop_duplicates(subset=['patient_id'])
patients_unique

,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,er,critical,12,True
2,PT-2002,Grace Mitchell,34,trauma,moderate,55,False
3,PT-2003,Daniel Brooks,28,er,high,40 mins,True
4,PT-2004,Nina Patel,61,cardiology,low,140,True
5,PT-2005,Marcus Johnson,47,trauma,moderate,60,False
6,PT-2006,Elena Cruz,26,er,critical,8,True
8,PT-2007,William Carter,73,icu,high,95,False
9,PT-2008,Sophia Nguyen,39,er,low,25,True
11,PT-2010,Sophia Nguyen,39,er,low,25,True
12,PT-2011,Ethan Walker,58,icu,critical,70,True


In [17]:
patients_unique['wait_time_minutes'] = pd.to_numeric(patients_unique['wait_time_minutes'], errors='coerce')
patients_unique

C:\Users\vesko\AppData\Local\Temp\ipykernel_21816\370191221.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patients_unique['wait_time_minutes'] = pd.to_numeric(patients_unique['wait_time_minutes'], errors='coerce')


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,er,critical,12.0,True
2,PT-2002,Grace Mitchell,34,trauma,moderate,55.0,False
3,PT-2003,Daniel Brooks,28,er,high,NaN,True
4,PT-2004,Nina Patel,61,cardiology,low,140.0,True
5,PT-2005,Marcus Johnson,47,trauma,moderate,60.0,False
6,PT-2006,Elena Cruz,26,er,critical,8.0,True
8,PT-2007,William Carter,73,icu,high,95.0,False
9,PT-2008,Sophia Nguyen,39,er,low,25.0,True
11,PT-2010,Sophia Nguyen,39,er,low,25.0,True
12,PT-2011,Ethan Walker,58,icu,critical,70.0,True


In [18]:
avg_wait_time = patients_unique['wait_time_minutes'].mean()
patients_unique['wait_time_minutes'] = patients_unique['wait_time_minutes'].fillna(avg_wait_time)
patients_unique

C:\Users\vesko\AppData\Local\Temp\ipykernel_21816\2119468426.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patients_unique['wait_time_minutes'] = patients_unique['wait_time_minutes'].fillna(avg_wait_time)


,patient_id,name,age,department,triage_level,wait_time_minutes,admitted
0,PT-2001,Anthony Ramirez,52,er,critical,12.000000,True
2,PT-2002,Grace Mitchell,34,trauma,moderate,55.000000,False
3,PT-2003,Daniel Brooks,28,er,high,54.444444,True
4,PT-2004,Nina Patel,61,cardiology,low,140.000000,True
5,PT-2005,Marcus Johnson,47,trauma,moderate,60.000000,False
6,PT-2006,Elena Cruz,26,er,critical,8.000000,True
8,PT-2007,William Carter,73,icu,high,95.000000,False
9,PT-2008,Sophia Nguyen,39,er,low,25.000000,True
11,PT-2010,Sophia Nguyen,39,er,low,25.000000,True
12,PT-2011,Ethan Walker,58,icu,critical,70.000000,True
